In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix


In [3]:
#######   Categorical Datasets ############

print("\n===== CATEGORICAL DATASET =====")

df_cat = pd.read_csv("iris_flower_categorical.csv")

X_cat = df_cat.drop("species", axis=1).values
y_cat = df_cat["species"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_cat, y_cat, test_size=0.2, random_state=42
)


===== CATEGORICAL DATASET =====


In [4]:

#  Numerical Gaussian Naive Bayes 
class GaussianNB:

    def fit(self, X, y):

        self.classes = np.unique(y)
        self.mean = {}
        self.var = {}
        self.prior = {}

        for c in self.classes:

            X_c = X[y == c]

            self.mean[c] = np.mean(X_c, axis=0)
            self.var[c] = np.var(X_c, axis=0)

            self.prior[c] = len(X_c) / len(X)


    def gaussian(self, class_idx, x):

        mean = self.mean[class_idx]
        var = self.var[class_idx]

        numo= np.exp(-(x-mean)**2 / (2*var))
        deno = np.sqrt(2*np.pi*var)

        return numo/ deno


    def predict(self, X):

        preds = []

        for x in X:

            posteriors = []

            for c in self.classes:

                prior = np.log(self.prior[c])

                likelihood = np.sum(np.log(self.gaussian(c, x)))

                posterior = prior + likelihood

                posteriors.append(posterior)

            preds.append(self.classes[np.argmax(posteriors)])

        return np.array(preds)

In [5]:
# Categorical Naive Bayes

class CategoricalNB:

    def fit(self, X, y):

        self.classes = np.unique(y)
        self.prior = {}
        self.likelihood = {}

        for c in self.classes:

            X_c = X[y == c]

            self.prior[c] = len(X_c) / len(X)

            self.likelihood[c] = []

            for col in range(X.shape[1]):

                values, counts = np.unique(X_c[:,col], return_counts=True)

                prob = {}

                for v,count in zip(values,counts):
                    prob[v] = count / len(X_c)

                self.likelihood[c].append(prob)


    def predict(self, X):

        preds = []

        for x in X:

            probs = {}

            for c in self.classes:

                prob = self.prior[c]

                for i,value in enumerate(x):

                    if value in self.likelihood[c][i]:
                        prob *= self.likelihood[c][i][value]
                    else:
                        prob *= 1e-6

                probs[c] = prob

            preds.append(max(probs, key=probs.get))

        return np.array(preds)

In [6]:

#############  Numerical Gaussian Naive Bayes ##################
print("\n===== Numerical Gaussian Naive Bayes =====")
gnb = GaussianNB()
gnb.fit(X_train1, y_train1)  # Using last X_train (numerical/categorical?) -> We'll fix

# Since you re-used X_train/y_train for categorical split, let's split numerical separately:
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X_num, y_num, test_size=0.2, random_state=42
)

gnb.fit(X_train1, y_train1)
y_pred_num = gnb.predict(X_test1)

print("Accuracy:", accuracy_score(y_test1, y_pred_num))
print("Confusion Matrix:\n", confusion_matrix(y_test1, y_pred_num))


# ##########  Categorical Naive Bayes ###########
print("\n===== Categorical Naive Bayes =====")
cnb = CategoricalNB()
cnb.fit(X_train, y_train)  # X_train/y_train from categorical split
y_pred_cat = cnb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_cat))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_cat))


===== Numerical Gaussian Naive Bayes =====
Accuracy: 1.0
Confusion Matrix:
 [[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]

===== Categorical Naive Bayes =====
Accuracy: 1.0
Confusion Matrix:
 [[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
